# Phase 2: 汎化性検証を全7試合で実施 — Colab

**目的**: J03WPY 1試合で確認した「PINN(単一の時間依存フィット)は、Baseline1(単一の定数フィット)より複数の$\tau$にわたって一貫して予測できる」という結果(`documents/phase2_pilot_results.md`)が、他の試合でも再現するか確認する。

実行するロジックは `scripts/phase2_generalization_check.py`(試合ごとにPINNを学習し、$\tau=0.5,1.0,1.5,2.0,2.5$秒でBaseline1と予測精度を比較)。

**実行前に**: メニューの `ランタイム > ランタイムのタイプを変更` で **GPU** を選択してください。

## 1. Google Drive をマウント(結果の永続化用)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUT_DIR = '/content/drive/MyDrive/pi-fsm/phase2_generalization_multi'
import os
os.makedirs(OUT_DIR, exist_ok=True)
print('results will be saved to', OUT_DIR)

## 2. リポジトリを取得

In [ ]:
%cd /content
if not os.path.exists('/content/pi-fsm'):
    !git clone https://github.com/ryu622/pi-fsm.git
%cd /content/pi-fsm
!git pull
!git log --oneline -5

## 3. 依存関係のインストール

In [ ]:
!pip install -q -e .

import torch
print('torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('WARNING: no GPU detected — go to ランタイム > ランタイムのタイプを変更 > GPU')

## 4. 全7試合で汎化性検証を実行

試合ごとにPINNを学習(1試合あたりローカルCPU/MPSで約8〜9分、GPUならもっと速いはず)。
中断しても完了済みの試合はスキップされるので再実行して問題ない。

In [ ]:
!python scripts/phase2_generalization_check.py --out-dir "{OUT_DIR}"

## 5. 結果の確認

In [ ]:
import pandas as pd
from IPython.display import Image, display

combined = pd.read_csv(f'{OUT_DIR}/all_matches_summary.csv')
print('PINN RMSE by match x tau:')
display(combined.pivot(index='tau', columns='match_id', values='rmse_pinn').round(3))
print('Baseline1 RMSE by match x tau:')
display(combined.pivot(index='tau', columns='match_id', values='rmse_baseline1').round(3))

# 判定: 各試合でPINNの誤差がBaseline1の誤差より一貫して小さいか
summary = combined.groupby('match_id').apply(
    lambda g: (g['rmse_pinn'] < g['rmse_baseline1']).mean(), include_groups=False
)
print('PINNがBaseline1を上回った tau の割合(試合ごと):')
display(summary)

display(Image(f'{OUT_DIR}/generalization_multi.png'))

## 次のステップ

- 全試合で同じ傾向(PINNの誤差がほぼ一定、Baseline1の誤差が$\tau$とともに増加)が見られれば、J03WPYの結果が一般的なものだと確認できる
- 結果はDriveの `{OUT_DIR}` に試合ごとのJSON + `all_matches_summary.csv` + 図として永続化される
- 確認できたら、フェーズ4(合成データでのリカバリーテスト、測定誤差 vs モデルの限界の切り分け)に進む